# ComfyUI on Colab (T4)

Runtime > Change runtime type > **T4 GPU** before running anything.

Run cells top to bottom. The last cell prints a public URL for the ComfyUI web interface.

**T4 limits that matter:** 16 GB VRAM, compute capability 7.5 — no bf16, no fp8, no fp4.
fp8_e4m3fn checkpoints need Ada (sm_89) or newer; nvfp4 needs Blackwell. On a T4, use fp16
or GGUF-quantised models. SDXL fits comfortably; 14B video models need GGUF Q4 and are slow.

Colab runtimes are ephemeral — the VM disk is wiped when the session ends. Mount Drive
(cell 3) if you want models and outputs to survive.

In [ ]:
# 1. Verify GPU. Fails loudly if no GPU or the wrong one is attached.
import subprocess, sys

out = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True)
assert out.returncode == 0, 'No GPU attached. Runtime > Change runtime type > T4 GPU.'
print(out.stdout.strip())

import torch
cap = torch.cuda.get_device_capability()
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'torch {torch.__version__}  sm_{cap[0]}{cap[1]}  {vram:.1f} GB VRAM')
if cap[0] < 8:
    print('WARNING: pre-Ampere GPU — no bf16/fp8. Use fp16 or GGUF weights only.')

In [ ]:
# 2. Install ComfyUI. Colab already ships a CUDA torch build, so torch is left alone.
%cd /content
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt

# ComfyUI-Manager makes installing custom nodes a UI action instead of a notebook edit.
!git clone --depth 1 https://github.com/ltdrdata/ComfyUI-Manager custom_nodes/ComfyUI-Manager
!pip install -q -r custom_nodes/ComfyUI-Manager/requirements.txt

# GGUF loader — the practical way to run large models on a 16 GB T4.
!git clone --depth 1 https://github.com/city96/ComfyUI-GGUF custom_nodes/ComfyUI-GGUF
!pip install -q gguf
print('ComfyUI installed')

In [ ]:
# 3. OPTIONAL — mount Drive so models and outputs persist across sessions.
# Skip this cell to keep everything on the ephemeral VM disk.
USE_DRIVE = True

import os, shutil

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    root = '/content/drive/MyDrive/comfyui'
    for sub in ['models/checkpoints', 'models/diffusion_models', 'models/text_encoders',
                'models/vae', 'models/loras', 'output']:
        os.makedirs(f'{root}/{sub}', exist_ok=True)
    # Point ComfyUI at Drive by replacing its local dirs with symlinks.
    for sub in ['models', 'output']:
        local = f'/content/ComfyUI/{sub}'
        if os.path.islink(local):
            os.unlink(local)
        elif os.path.isdir(local):
            shutil.rmtree(local)
        os.symlink(f'{root}/{sub}', local)
    print(f'models and output now live in {root}')
else:
    print('using ephemeral VM storage — everything is lost when the runtime ends')

In [ ]:
# 4. Download models. Edit this list — these are examples, not a fixed set.
# Anything already present in Drive is skipped.
MODELS = [
    # (destination folder, filename, url)
    ('checkpoints', 'sd_xl_base_1.0.safetensors',
     'https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors'),
]

# Private/gated repos need a token: HF_TOKEN = 'hf_...'
HF_TOKEN = ''

import os, subprocess

for folder, name, url in MODELS:
    dest_dir = f'/content/ComfyUI/models/{folder}'
    os.makedirs(dest_dir, exist_ok=True)
    dest = f'{dest_dir}/{name}'
    if os.path.exists(dest):
        print(f'skip {name} — already present')
        continue
    cmd = ['wget', '-q', '--show-progress', '-O', dest, url]
    if HF_TOKEN:
        cmd[2:2] = ['--header', f'Authorization: Bearer {HF_TOKEN}']
    print(f'downloading {name}')
    subprocess.run(cmd, check=True)

!du -sh /content/ComfyUI/models/* 2>/dev/null

In [ ]:
# 5. Launch ComfyUI behind a cloudflare tunnel and print the public URL.
import re, subprocess, time, threading, atexit

!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

comfy = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188', '--lowvram'],
    cwd='/content/ComfyUI',
    stdout=open('/content/comfyui.log', 'w'), stderr=subprocess.STDOUT)
atexit.register(comfy.terminate)

# Wait for the server to bind before tunnelling to it.
for _ in range(120):
    time.sleep(1)
    if 'To see the GUI go to' in open('/content/comfyui.log').read():
        break
else:
    print(open('/content/comfyui.log').read()[-3000:])
    raise RuntimeError('ComfyUI did not start — see log above')
print('ComfyUI is up on 127.0.0.1:8188')

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
atexit.register(tunnel.terminate)

for line in tunnel.stdout:
    m = re.search(r'https://[-\w]+\.trycloudflare\.com', line)
    if m:
        print('\nOpen ComfyUI here:', m.group(0), '\n')
        break

# Keep this cell running — closing it kills both the server and the tunnel.
# Server log tail, refreshed every 30s:
while comfy.poll() is None:
    time.sleep(30)
    print(open('/content/comfyui.log').read()[-500:], flush=True)

## Notes

- The `trycloudflare.com` URL is public but unguessable, and dies with the tunnel.
  Anyone holding it can drive your ComfyUI, so don't post it.
- `--lowvram` keeps the model off the GPU until each node needs it. Drop it if a model
  fits in 16 GB comfortably; keep it for anything near the limit.
- Free-tier Colab has historically blocked image-generation web UIs. If the runtime is
  killed shortly after launch, that is the cause, and Pro is the fix.
- Outputs land in `ComfyUI/output` — under Drive if cell 3 ran, otherwise gone at session end.